# 11b — Validation regimes v2 (eligibility-based, identity-bound)

Replaces the v1 regime table of notebook 11 for Gate A. v1 split eligible truths 34 / 33 / 33 by a hash, used `test_simulated_strict` for C1, and was built against the TRAIN-only universe (274,195 candidates) — and notebook 13 never checked its signature.

**v2 assignment, per truth connectivity (natural counts, no fractions, no random split):**

| regime | rule | authority |
|---|---|---|
| **C1** | ≥ 1 eligible **mirror-aware** reference of the truth | exact lazy-walk QCR flag `has_ge1_mirror_aware` (truth row) |
| **C2** | no mirror-aware reference, **externally reachable** | universe provenance minus TRAIN (`validation_v2.external_sources`: COCONUT; PUBCHEM only once its index exists) |
| **C3** | neither | — |
| **UNRESOLVED** | no mirror-aware evidence for any query of the connectivity | shown explicitly, never folded into C2 / C3 |

`test_simulated_strict` is a **sensitivity diagnostic only**. Panels: ALL_DEV, TIMSTOF_DEV (instrument diagnostic, formerly `host_like`), HOST (file-backed enveda-np-examples), NP_PANEL (structure-only NP-likeness with a user threshold).

The manifest binds the table to: git commit, protocol version, dev-query and split identity, universe build id (modern or **adopted**), external source set, membership identity, mirror-aware protocol + evidence identity, panel manifest identity, assignment policy and code identity, seed, config. Notebook 13 refuses to run on any mismatch (`REGIME_REBUILD_REQUIRED`).

Run order: 10 → 11a (legacy adoption) → **11b** → 13. CPU only. The old `validation_regimes.parquet` is **copied** to `data/processed/legacy/validation_regimes_pre_external_universe.parquet` and marked STALE (never deleted).

In [ ]:
# bootstrap: Drive, repository (cloned by notebook 10), config -- nothing is computed yet
import os, sys, json
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
_repo_candidates = [os.environ.get('ENVEDA_REPO_ROOT'), str(Path('/content', 'Enveda')), str(DRIVE_ROOT / 'repo')]
REPO_ROOT = next((Path(p) for p in _repo_candidates if p and (Path(p) / 'src' / 'casmi').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'CASMI repository not found in {_repo_candidates}: run notebook 10 or set ENVEDA_REPO_ROOT')
os.environ['ENVEDA_REPO_ROOT'] = str(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from casmi.workspace.environment import ensure_packages, set_seeds, system_report
ensure_packages({'rdkit': 'rdkit', 'yaml': 'pyyaml'})
from casmi.workspace.config import load_v2_config, input_path
CFG, P = load_v2_config(REPO_ROOT / 'configs' / 'casmi_v2_colab.yaml')
P.ensure()
SEED = set_seeds(CFG['runtime']['seed'])
NOTEBOOK = '11b_colab_validation_regimes_v2'
V2 = CFG['validation_v2']
EXT_SOURCES = list(V2['external_sources'])
UNIVERSE_ROOT = P.candidate_db_dir / 'universe'
print('external sources:', EXT_SOURCES, '| universe', UNIVERSE_ROOT)
print(system_report())

In [ ]:
# universe identity: modern manifest v3 or an ADOPTED legacy universe (notebook 11a); anything else stops here
from casmi.candidates.legacy_adoption import effective_universe_identity
UID = effective_universe_identity(UNIVERSE_ROOT)
print(json.dumps({k: UID.get(k) for k in ('kind', 'build_id', 'universe_status', 'external_sources', 'problems')}, indent=2))
if UID['kind'] not in ('modern', 'legacy_adopted'):
    raise RuntimeError(f"universe identity {UID['kind']!r} {UID.get('problems')}: run notebook 11a (legacy adoption) first")
MISSING_SRC = [s for s in EXT_SOURCES if s != 'PUBCHEM' and s not in UID['external_sources']]
if MISSING_SRC:
    raise RuntimeError(f'validation_v2.external_sources {MISSING_SRC} have no candidates in the universe {UID["external_sources"]}')

In [ ]:
# grouped validation queries, connectivity folds, reference-library connectivities (the visible test is never used)
import numpy as np, pandas as pd
DEV_PATH, FOLDS_PATH = input_path(CFG, P, 'dev_queries'), input_path(CFG, P, 'connectivity_folds')
dev = pd.read_parquet(DEV_PATH)
dev['query_id'] = dev['query_id'].astype(str)
folds = pd.read_parquet(FOLDS_PATH)
tsm = pd.read_parquet(input_path(CFG, P, 'train_spectrum_metadata'), columns=['train_spectrum_id', 'connectivity_key'])
REF_CONN = tsm['connectivity_key'].astype(str).to_numpy()
not_train = ~dev['query_id'].isin(set(tsm['train_spectrum_id'].astype(str)))
assert not not_train.any(), f'{int(not_train.sum())} dev queries are not training spectra'
del tsm
print(f'dev queries {len(dev):,} | truth connectivities {dev["true_connectivity_key"].nunique():,} | reference spectra {len(REF_CONN):,}')

In [ ]:
# NP_PANEL: structure-only NP-likeness of the dev truths (TRAIN structure-table SMILES); only when a threshold is declared
from casmi.validation.panels import PanelConfig, ensure_np_scores, np_method_status
PCFG = PanelConfig.from_dict(CFG['panels'])
print('NP method:', PCFG.np_method, '| threshold:', PCFG.np_threshold, '| provenance:', PCFG.np_threshold_provenance)
if PCFG.np_threshold is None:
    print('NP_THRESHOLD_NOT_SET: NP_PANEL stays NOT_AVAILABLE (set panels.NP_PANEL.threshold BEFORE running, never after results)')
    NP_SCORES, NP_INFO = None, None
else:
    print('method status:', np_method_status(PCFG.np_method))
    st = pd.read_parquet(input_path(CFG, P, 'structure_table'), columns=['connectivity_key', 'canonical_smiles']).dropna()
    st = st.assign(connectivity_key=st['connectivity_key'].astype(str)).drop_duplicates('connectivity_key').set_index('connectivity_key')
    st = st.loc[st.index.isin(set(dev['true_connectivity_key'].astype(str))), 'canonical_smiles']
    NP_SCORES, NP_INFO = ensure_np_scores(PCFG, P.drive_root, st)
    print(f'NP scores: {NP_SCORES["np_score"].notna().sum():,} scored / {len(NP_SCORES):,} truths |', NP_INFO)

In [ ]:
# four panels (never merged); HOST from its file (missing -> HOST_PANEL_SOURCE_MISSING); decision panel HOST, else NP_PANEL
from casmi.validation.panels import panels_for_run
from casmi.validation.external_coverage import write_json
PAN = panels_for_run(CFG, P.drive_root, dev)
Q, PANEL_STATUS, DECISION_PANEL, PANEL_MANIFEST = PAN['queries'], PAN['status'], PAN['decision'], PAN['manifest']
display(pd.DataFrame(PANEL_STATUS).T[['status', 'role', 'reason', 'n_queries', 'n_molecules']])
print('decision panel:', DECISION_PANEL['decision_panel'], '|', DECISION_PANEL['reason'])
print('panel_manifest_identity:', PANEL_MANIFEST['panel_manifest_identity'])
write_json(P.drive_root / V2['panel_manifest_path'], PANEL_MANIFEST)

In [ ]:
# C1 authority: mirror-aware truth evidence (exact QCR flags); test_simulated_strict = sensitivity only
from casmi.validation.regimes_v2 import load_reference_evidence
ME, SE = V2['mirror_aware_evidence'], V2['test_simulated_strict_evidence']
MIRROR, MIRROR_INFO = load_reference_evidence([P.drive_root / f for f in ME['files']], ME['flag_column'], 'mirror_aware')
STRICT, STRICT_INFO = load_reference_evidence([P.drive_root / f for f in SE['files']], SE['flag_column'], 'test_simulated_strict')
print(json.dumps({'mirror_aware': MIRROR_INFO, 'test_simulated_strict': STRICT_INFO}, indent=2, default=str))
if not MIRROR_INFO['available']:
    raise FileNotFoundError(f"MIRROR_AWARE_EVIDENCE_MISSING: upload {ME['files']} (local data/processed) to Drive -- C1 cannot be assigned")

In [ ]:
# leakage-free external reachability of every truth (TRAIN provenance never counts)
from casmi.candidates.pubchem_membership import PubChemMembershipIndex
from casmi.validation.external_coverage import build_external_membership, universe_truth_rows
from casmi.validation.regimes_v2 import external_reachability
POP_COLS = ['query_id', 'true_connectivity_key', 'fold', 'population']
HOST_DF = PAN['host_frame']
POP = pd.concat([dev.assign(population='ALL_DEV')[POP_COLS]] + ([HOST_DF.assign(population='HOST')[POP_COLS]] if HOST_DF is not None else []),
                ignore_index=True)
KEYS = POP['true_connectivity_key'].astype(str).unique()
TRUTH_ROWS = universe_truth_rows(UNIVERSE_ROOT, KEYS)
PC_DIR = P.drive_root / CFG['coverage']['pubchem_membership']['index_dir']
PCI = PubChemMembershipIndex(PC_DIR) if ('PUBCHEM' in EXT_SOURCES and PubChemMembershipIndex.available(PC_DIR)) else None
if 'PUBCHEM' in EXT_SOURCES and PCI is None:
    raise FileNotFoundError(f'PUBCHEM_MEMBERSHIP_SOURCE_MISSING: PUBCHEM is listed in external_sources but {PC_DIR} has no index')
MEMB = external_reachability(build_external_membership(KEYS, TRUTH_ROWS, PCI), EXT_SOURCES)
print(f'truths {len(KEYS):,} | in universe {int(MEMB["membership_in_full_train_union"].sum()):,} | externally reachable {int(MEMB["external_reachable"].sum()):,}')

In [ ]:
# eligibility-based assignment (natural counts) + definitional invariants + per-(panel, connectivity) records
from casmi.validation.regimes_v2 import assert_regime_v2_consistency, assign_regimes_v2, molecule_records, regime_counts_by_panel
REG, MOL = assign_regimes_v2(POP, MIRROR, MEMB, EXT_SOURCES, strict_evidence=STRICT)
CONS = assert_regime_v2_consistency(MOL)
RECORDS = molecule_records(Q, MOL)
COUNTS = regime_counts_by_panel(RECORDS)
display(CONS)
display(COUNTS.pivot(index='panel', columns='regime', values='n_molecules').fillna(0).astype(int))
print(MOL['regime_reason'].value_counts().to_string())
SENS = pd.crosstab(MOL['has_mirror_aware_reference'].astype(str), MOL['has_test_simulated_strict_reference'].astype(str))
print('mirror-aware (rows) vs test_simulated_strict (columns; <NA> = no sensitivity evidence):')
display(SENS)

In [ ]:
# leakage + masking invariants on ALL_DEV, every fold (v1 checks reused; raises on any failure)
from casmi.candidates.universe import load_candidate_keys
from casmi.validation.regimes import RegimeConfig, assert_regime_integrity
UKEYS = load_candidate_keys(UNIVERSE_ROOT)
DEVREG = REG.loc[REG['population'] == 'ALL_DEV', ['query_id', 'true_connectivity_key', 'fold', 'regime']]
CHECKS = pd.concat([assert_regime_integrity(DEVREG, folds, REF_CONN, UKEYS, f, RegimeConfig()) for f in sorted(DEVREG['fold'].unique())],
                   ignore_index=True)
display(CHECKS)
print('ALL REGIME CHECKS PASSED:', bool(CHECKS['passed'].all()))

In [ ]:
# identity + signature (the SAME function notebook 13 uses to verify)
from casmi.validation.regimes_v2 import check_regime_manifest, regime_identity_from_run, regime_signature
IDENTITY = regime_identity_from_run(cfg=CFG, repo_root=REPO_ROOT, dev=dev, dev_path=DEV_PATH, folds=folds, folds_path=FOLDS_PATH,
                                    universe_identity=UID, mirror_info=MIRROR_INFO,
                                    panel_manifest_identity=PANEL_MANIFEST['panel_manifest_identity'], pubchem_index=PCI,
                                    host_path=PAN['host_path'])
SIG = regime_signature(IDENTITY)
MAN_PATH = P.drive_root / V2['manifest_path']
PRE = check_regime_manifest(MAN_PATH, IDENTITY, V2['strict_git_commit'])
print('git commit', IDENTITY['git_commit'], '| universe', IDENTITY['universe_build_id'], '| external', IDENTITY['external_sources'])
print('regime signature', SIG, '| persisted manifest before this run:', PRE['status'], [m['field'] for m in PRE['mismatches']])
if not IDENTITY['git_commit']:
    print('WARNING: git commit unknown (repository without .git) -- experiments bound to these regimes will be refused')

In [ ]:
# archive the stale v1 table (copy + STALE marker), then write the v2 table, records, manifest and exclusion lists
from casmi.validation.regimes_v2 import archive_legacy_regimes, write_exclusion_manifests, write_regime_manifest
ARCHIVE = archive_legacy_regimes(P.processed_dir, P.drive_root / V2['legacy_archive_dir'])
print(json.dumps(ARCHIVE, indent=2, default=str))
REG.to_parquet(P.drive_root / V2['regimes_path'], index=False)
RECORDS.to_parquet(P.drive_root / V2['records_path'], index=False)
EXCL = write_exclusion_manifests(RECORDS, P.drive_root / V2['exclusion_dir'], SIG, PANEL_MANIFEST['panel_manifest_identity'])
COUNT_SUMMARY = {'by_panel': COUNTS.to_dict(orient='records'), 'by_reason': MOL['regime_reason'].value_counts().to_dict(),
                 'n_connectivities': int(len(MOL)), 'n_queries': int(len(REG))}
MANIFEST = write_regime_manifest(MAN_PATH, IDENTITY, COUNT_SUMMARY, extra={
    'notebook': NOTEBOOK, 'panel_status': PANEL_STATUS, 'decision_panel': DECISION_PANEL, 'mirror_evidence': MIRROR_INFO,
    'strict_evidence': STRICT_INFO, 'integrity_checks_passed': bool(CHECKS['passed'].all()), 'archive': ARCHIVE,
    'exclusions': EXCL, 'files': {'regimes': V2['regimes_path'], 'records': V2['records_path']}})
print('saved regime manifest', MAN_PATH, '| signature', MANIFEST['regime_signature'])

In [ ]:
# natural regime shares per panel (no fractions were imposed)
import matplotlib.pyplot as plt
FIG_DIR = P.reports_dir / 'validation'
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(1, 1, figsize=(9, 4))
COUNTS.pivot(index='panel', columns='regime', values='share').fillna(0).plot.bar(stacked=True, ax=ax, title='natural regime shares per panel (v2)')
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(FIG_DIR / 'regime_shares_v2.png', dpi=120)

## Next

* Run **13_candidate_coverage**. It recomputes this identity and stops with `REGIME_REBUILD_REQUIRED` on any mismatch.
* Re-run 11b whenever an input changes: universe (re-adoption / rebuild), `external_sources` (e.g. PUBCHEM added after `scripts/build_pubchem_membership_index.py`), mirror-aware evidence files, panels (HOST file, NP threshold), dev queries or folds.
* `UNRESOLVED` connectivities have no mirror-aware evidence (query not walked by QCR, or truth outside the walked pool). They are reported in every panel and never folded into C2 / C3.